# Clasificar las citas no usadas con el QLoRA (subconjunto por subconjunto)

Aplica el adaptador **QLoRA v2** (`qlora_citas_adapter_v2/`, entrenado en `finetune_qlora_citas.ipynb`) a cada subconjunto generado por `proyecto_de_grado/scripts/particionar_citas_no_usadas.py` (`subconjuntos_qlora/CITAS_NO_USADAS_NN.csv`). **No se entrena nada**: solo inferencia.

Por cada subconjunto:
- se clasifica con *checkpoint* en un `.jsonl` (reanudable: si se interrumpe, vuelve a ejecutar la celda y continúa),
- se exporta un CSV con el mismo formato que `ANOTACION_18000_CITAS_REALES_QLORA_v2.csv`: `subconjuntos_qlora/CITAS_NO_USADAS_NN_QLORA_v2.csv`,
- se muestra cuántas citas quedaron por clase en ese subconjunto y el **acumulado** (incluye las 18 000 ya clasificadas) frente a la meta de 2 000 por clase.

> Las etiquetas son **pre-anotación del modelo**, no gold humano. Libera la GPU antes (`ollama stop qwen2.5:7b-instruct`) y usa el kernel "Python (qlora)".

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import json, time
from pathlib import Path
import pandas as pd

BASE = Path('.')
SUBSET_DIR   = BASE / 'subconjuntos_qlora'
ADAPTER_DIR  = BASE / 'qlora_citas_adapter_v2'          # adaptador validado
PREV_CSV     = BASE / 'ANOTACION_18000_CITAS_REALES_QLORA_v2.csv'   # ya clasificadas: cuentan para la meta
BASE_MODEL   = 'Qwen/Qwen2.5-7B-Instruct'

META_POR_CLASE = 2000
ABSTRACT_CHARS = 400
INFER_BATCH = 8
CONF_HIGH, CONF_MID = 0.80, 0.50     # mismos umbrales que el notebook de entrenamiento

FUNCTIONS = {
    'Background': 'Antecedentes generales del tema: contexto, trasfondo o historia de un campo; revisa literatura existente.',
    'Gap': 'Identifica un vacío o limitación en el trabajo previo y justifica la elección del tema.',
    'Basis': 'Fundamenta una afirmación o premisa; punto de partida sobre el que el trabajo actual construye o expande.',
    'Comparison': 'Compara explícitamente enfoques: el trabajo actual con los citados, o los citados entre sí.',
    'Application': 'Utiliza directamente, SIN modificar, un método, técnica, herramienta o dato del trabajo citado.',
    'Improvement / Modification': 'Adapta, mejora o extiende el método o herramienta citados.',
    'Evidence': 'Sustenta con evidencia o resultados: respalda afirmaciones, hipótesis o hallazgos.',
    'Identification of the Originator': 'Atribuye el origen de una idea, concepto, método o teoría a la obra pionera.',
    'Further Reading': 'Invita a leer más detalles: remite a fuentes externas para mayor profundidad.',
}
LABELS = list(FUNCTIONS) + ['Ambiguo', 'No evaluable']      # el orden DEBE ser el del entrenamiento
LETTERS = 'ABCDEFGHIJK'[:len(LABELS)]
L2LET = dict(zip(LABELS, LETTERS))
COLS = {'ctx': 'Contexto TARGETCIT', 'title': 'Título citado', 'abs': 'Resumen citado',
        'ev': '¿Evaluable?', 'fn': 'Función principal', 'conf': 'Confianza', 'just': 'Justificación / duda'}

SYSTEM_PROMPT = ('Clasifica la función de la cita marcada TARGETCIT (OTHERCIT son otras citas) en un artículo de lingüística computacional. '
                 'Responde solo con la letra.\n'
                 + '\n'.join(f'{L2LET[k]}) {k}' for k in LABELS))

def user_message(row):
    title = row[COLS['title']].strip() or '(sin título)'
    return f"CONTEXTO:\n{row[COLS['ctx']].strip()}\n\nTRABAJO CITADO: {title}\nRESUMEN: {row[COLS['abs']].strip()[:ABSTRACT_CHARS]}"

subset_files = sorted(SUBSET_DIR.glob('CITAS_NO_USADAS_[0-9][0-9].csv'))
assert subset_files, f'No hay subconjuntos en {SUBSET_DIR}. Ejecuta proyecto_de_grado/scripts/particionar_citas_no_usadas.py'
assert ADAPTER_DIR.exists(), f'Falta el adaptador {ADAPTER_DIR}'
print(len(subset_files), 'subconjuntos:', [f.name for f in subset_files])

5 subconjuntos: ['CITAS_NO_USADAS_01.csv', 'CITAS_NO_USADAS_02.csv', 'CITAS_NO_USADAS_03.csv', 'CITAS_NO_USADAS_04.csv', 'CITAS_NO_USADAS_05.csv']


## 1. Cargar modelo base + adaptador (solo inferencia)

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), 'No hay GPU CUDA visible'
print(torch.cuda.get_device_name(0), f'| VRAM libre: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
LETTER_IDS = [tok.encode(l, add_special_tokens=False) for l in LETTERS]
assert all(len(i) == 1 for i in LETTER_IDS), LETTER_IDS
LETTER_IDS = [i[0] for i in LETTER_IDS]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()
print('Adaptador cargado desde', ADAPTER_DIR, f'| VRAM libre: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA GeForce RTX 4070 SUPER | VRAM libre: 10.8 GiB


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:50<00:00,  6.77it/s]


Adaptador cargado desde qlora_citas_adapter_v2 | VRAM libre: 5.3 GiB


## 2. Funciones de inferencia (idénticas a las del notebook de entrenamiento)

In [3]:
def prompt_text(row):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_message(row)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def predict_rows(rows):
    enc = tok([prompt_text(r) for r in rows], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
    try:
        logits = model(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :]
    except TypeError:
        logits = model(**enc, use_cache=False).logits[:, -1, :]
    probs = logits[:, LETTER_IDS].float().softmax(-1).cpu()
    out = []
    for p in probs:
        k = int(p.argmax()); out.append({'label': LABELS[k], 'p': float(p[k]), 'dist': [round(float(x), 4) for x in p]})
    return out

def to_record(cid, pr):
    lab, p = pr['label'], pr['p']
    return {'context_id': cid, 'label': lab, 'p': round(p, 4), 'dist': pr['dist'],
            'evaluable': 'Evaluable' if lab in FUNCTIONS else lab,
            'funcion': lab if lab in FUNCTIONS else '',
            'confianza': 'Alta' if p >= CONF_HIGH else 'Media' if p >= CONF_MID else 'Baja'}

def load_done(path):
    done = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                d = json.loads(line); done[d['context_id']] = d
    return done

def run_batch(df, path, max_rows=None):
    done = load_done(path)
    todo = df[~df['context_id'].isin(done)].copy()
    todo['_len'] = todo[COLS['ctx']].str.len() + todo[COLS['abs']].str.len().clip(upper=ABSTRACT_CHARS)
    todo = todo.sort_values('_len')                      # lotes de longitud parecida = menos padding
    if max_rows: todo = todo.head(max_rows)
    rows = [r for _, r in todo.iterrows()]
    print(f'{len(done)} ya hechos, {len(rows)} por procesar')
    t0 = time.time()
    with path.open('a', encoding='utf-8') as f:
        for i in range(0, len(rows), INFER_BATCH):
            chunk = rows[i:i + INFER_BATCH]
            for r, pr in zip(chunk, predict_rows(chunk)):
                f.write(json.dumps(to_record(r['context_id'], pr), ensure_ascii=False) + '\n')
            f.flush()
            n = i + len(chunk)
            if (i // INFER_BATCH) % 25 == 0:
                rate = n / (time.time() - t0)
                print(f'{n}/{len(rows)}  {rate:.1f} filas/s  ETA {(len(rows) - n) / rate / 60:.0f} min')
    return load_done(path)

def conf(p): return 'Alta' if p >= CONF_HIGH else 'Media' if p >= CONF_MID else 'Baja'

def export_csv(df, results, out_csv):
    # mismo formato que ANOTACION_18000_CITAS_REALES_QLORA_v2.csv
    out = df.copy()
    m = out['context_id'].map(results)
    out[COLS['ev']]   = m.map(lambda d: d['evaluable'] if isinstance(d, dict) else '')
    out[COLS['fn']]   = m.map(lambda d: d['funcion'] if isinstance(d, dict) else '')
    out[COLS['conf']] = m.map(lambda d: conf(d['p']) if isinstance(d, dict) else '')
    out[COLS['just']] = m.map(lambda d: f"QLoRA p={d['p']:.2f}" if isinstance(d, dict) else '')
    out['Estado']     = m.map(lambda d: ('COMPLETO' if d['evaluable'] == 'Evaluable' else 'REGISTRADO') if isinstance(d, dict) else '')
    out.to_csv(out_csv, index=False, encoding='utf-8-sig')
    return out

def clase(df):
    # clase final = función si es evaluable; si no, 'Ambiguo' / 'No evaluable'; '' = sin clasificar
    return df[COLS['fn']].where(df[COLS['fn']] != '', df[COLS['ev']])

def conteo(df):
    return clase(df).value_counts().reindex(LABELS + ['']).fillna(0).astype(int).rename(index={'': 'sin clasificar'})

## 3. Prueba corta de velocidad (opcional)

Clasifica 64 citas del primer subconjunto para estimar filas/s. Los resultados quedan en su `.jsonl` y la corrida completa los reutiliza.

In [4]:
_df = pd.read_csv(subset_files[0], encoding='utf-8-sig', dtype=str, keep_default_na=False)
_ = run_batch(_df, subset_files[0].with_name(subset_files[0].stem + '_qlora_v2_preds.jsonl'), max_rows=64)

0 ya hechos, 64 por procesar
8/64  4.3 filas/s  ETA 0 min


## 4. Clasificar subconjunto por subconjunto

Edita `SUBCONJUNTOS` para elegir cuáles correr (por defecto, todos). Cada uno es reanudable. Al terminar cada subconjunto se escribe su CSV y se imprime su conteo por clase.

In [5]:
SUBCONJUNTOS = None        # None = todos; o p. ej. [1, 2] para solo CITAS_NO_USADAS_01 y _02

def num(p): return int(p.stem.rsplit('_', 1)[1])
tandas = [p for p in subset_files if SUBCONJUNTOS is None or num(p) in SUBCONJUNTOS]
conteos = {}

for sub in tandas:
    nombre = sub.stem                                           # CITAS_NO_USADAS_01
    jsonl = SUBSET_DIR / f'{nombre}_qlora_v2_preds.jsonl'
    out_csv = SUBSET_DIR / f'{nombre}_QLORA_v2.csv'
    df = pd.read_csv(sub, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    print(f'\n===== {nombre}: {len(df)} citas =====')
    results = run_batch(df, jsonl)
    out = export_csv(df, results, out_csv)
    sin = int((clase(out) == '').sum())
    assert sin == 0, f'{sin} filas sin clasificar en {nombre}'
    conteos[nombre] = conteo(out)
    print(f'-> {out_csv}')
    print(conteos[nombre].to_string())


===== CITAS_NO_USADAS_01: 8603 citas =====
64 ya hechos, 8539 por procesar
8/8539  6.7 filas/s  ETA 21 min
208/8539  7.6 filas/s  ETA 18 min
408/8539  7.6 filas/s  ETA 18 min
608/8539  7.7 filas/s  ETA 17 min
808/8539  7.7 filas/s  ETA 17 min
1008/8539  7.7 filas/s  ETA 16 min
1208/8539  7.8 filas/s  ETA 16 min
1408/8539  7.8 filas/s  ETA 15 min
1608/8539  7.8 filas/s  ETA 15 min
1808/8539  7.8 filas/s  ETA 14 min
2008/8539  7.8 filas/s  ETA 14 min
2208/8539  7.8 filas/s  ETA 13 min
2408/8539  7.8 filas/s  ETA 13 min
2608/8539  7.8 filas/s  ETA 13 min
2808/8539  7.8 filas/s  ETA 12 min
3008/8539  7.9 filas/s  ETA 12 min
3208/8539  7.8 filas/s  ETA 11 min
3408/8539  7.9 filas/s  ETA 11 min
3608/8539  7.9 filas/s  ETA 10 min
3808/8539  7.9 filas/s  ETA 10 min
4008/8539  7.9 filas/s  ETA 10 min
4208/8539  7.9 filas/s  ETA 9 min
4408/8539  7.9 filas/s  ETA 9 min
4608/8539  7.9 filas/s  ETA 8 min
4808/8539  7.9 filas/s  ETA 8 min
5008/8539  7.9 filas/s  ETA 7 min
5208/8539  7.9 filas/s  ET

## 5. Resumen: citas por clase y avance hacia la meta de 2 000

Se puede ejecutar en cualquier momento: lee los CSV ya exportados. Columna `18000` = las citas ya clasificadas antes; `faltan` = meta − total (0 si ya se alcanzó).

In [6]:
cols = {}
if PREV_CSV.exists():
    cols['18000'] = conteo(pd.read_csv(PREV_CSV, encoding='utf-8-sig', dtype=str, keep_default_na=False))
else:
    print('AVISO: no se encontró', PREV_CSV, '- el acumulado no incluye las 18 000')
for sub in subset_files:
    f = SUBSET_DIR / f'{sub.stem}_QLORA_v2.csv'
    if f.exists():
        cols[sub.stem.replace('CITAS_NO_USADAS_', 'sub')] = conteo(pd.read_csv(f, encoding='utf-8-sig', dtype=str, keep_default_na=False))

res = pd.DataFrame(cols)
res['TOTAL'] = res.sum(axis=1)
res['meta'] = [META_POR_CLASE if c in FUNCTIONS else None for c in res.index]
res['faltan'] = [max(0, META_POR_CLASE - t) if c in FUNCTIONS else None for c, t in zip(res.index, res['TOTAL'])]
res.loc['TOTAL citas'] = res.drop(columns=['meta', 'faltan']).sum().tolist() + [None, None]
display(res)
alc = [c for c in FUNCTIONS if res.loc[c, 'TOTAL'] >= META_POR_CLASE]
print(f'Clases que alcanzan {META_POR_CLASE}: {len(alc)}/{len(FUNCTIONS)}')
print('Clases por debajo de la meta:', {c: int(res.loc[c, 'TOTAL']) for c in FUNCTIONS if c not in alc})

,18000,sub01,sub02,sub03,sub04,sub05,TOTAL,meta,faltan
Función principal,,,,,,,,,
Background,7227,3427,3445,3533,3370,3390,24392,2000.0,0.0
Gap,695,313,357,340,354,335,2394,2000.0,0.0
Basis,935,433,411,420,448,398,3045,2000.0,0.0
Comparison,1926,902,882,906,898,929,6443,2000.0,0.0
Application,4596,2241,2277,2181,2209,2249,15753,2000.0,0.0
Improvement / Modification,338,184,165,182,167,175,1211,2000.0,789.0
Evidence,1196,586,571,553,603,606,4115,2000.0,0.0
Identification of the Originator,449,215,232,205,225,237,1563,2000.0,437.0
Further Reading,359,167,153,163,173,154,1169,2000.0,831.0


Clases que alcanzan 2000: 6/9
Clases por debajo de la meta: {'Improvement / Modification': 1211, 'Identification of the Originator': 1563, 'Further Reading': 1169}
